In [ ]:
# importing the  libraries
import pandas as pd
import numpy as np

# Loading all three CSVs
desc = pd.read_csv("Data files/product_descriptions.csv", delimiter=";")
props = pd.read_csv("Data files/product_properties.csv", delimiter=";")
manu = pd.read_csv("Data files/manufacturers.csv", delimiter=";")

# --- EDA for product_descriptions---
print(" product_descriptions.csv")
print("Shape:", desc.shape)
print("Columns:", desc.columns.tolist())
print(desc.head())
print(desc.isnull().sum())
print(desc.nunique())

# --- EDA for product_properties---
print("\n product_properties.csv")
print("Shape:", props.shape)
print("Columns:", props.columns.tolist())
print(props.head())
print(props.isnull().sum())
print(props.nunique())

# --- EDA for manufacturers ---
print("\n manufacturers.csv")
print("Shape:", manu.shape)
print("Columns:", manu.columns.tolist())
print(manu.head())
print(manu.isnull().sum())
print(manu.nunique())


In [ ]:
# Normalize "bad" values 
BAD_VALUES = ["n/a", "na", "none", "null", "", " ", "-", "N/A", "None"]

def clean_data(df, key_columns):
    df = df.replace(BAD_VALUES, np.nan)
    df = df.dropna(subset=key_columns) # Drop records with null join keys
    df = df.drop_duplicates()  #optional just to check
    return df


# Cleaning dataframes
descriptions = clean_data(desc, ["Articlenumber"])
properties = clean_data(props, ["Articlenumber", "Manufacturernumber"])
manufacturers = clean_data(manu, ["Manufacturernumber"])

# Merging datasets
merged = properties.merge(manufacturers, on="Manufacturernumber", how="inner")
merged = merged.merge(descriptions, on="Articlenumber", how="inner")

# Saving cleaned dataset
merged.to_csv("output/cleaned_catalog.csv", index=False)
print(" Cleaned catalog saved to 'output/cleaned_catalog.csv'")


In [ ]:
# loading data from output of the data pipeline from the first part of the assignment


from sqlalchemy import create_engine #Connecting to SQL Database using SQLAlchemy 


# MySQL Connection Parameters
username = "DB_USERNAME"               
password = "DB_PASSWORD"     
host = "DB_HOST"             
port = "DB_PORT"                  
database = "DB_NAME"           

# Creating SQLAlchemy connection string
connection_string = f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"

# Creating SQLAlchemy Engine
engine = create_engine(connection_string)

# Loading DataFrame into MySQL
merged.to_sql(name="clean_catalog", con=engine, if_exists="fail", index=False)

print("Table created and data loaded into MySQL.")
